In [10]:
import pandas as pd
import numpy as np

import sys
sys.path.append('../')

from scripts.tree_node import NewTreeNode
from scripts.tree_analysis import (
    get_estimated_rewards,
    TABLETOP_TRUE_WEIGHTS,
    TABLETOP_FEATURE_BOUNDS,
    GRASS_STREET_NAV_TRUE_WEIGHTS,
    GRASS_STREET_NAV_FEATURE_BOUNDS,
)
from typing import List, Optional

import os
import pickle
import ast

In [11]:
data = pd.read_csv("../../server/survey_export_pilot_3.csv")

In [12]:
survey_metrics = ['mental_demand', 'success_level', 'frustration_level', 
                  'trajectory_choice_ease', 'difference_clarity', 'preference_learning']
    
# First filter by expected choice counts per test_type
if 'tabletop' in data['test_type'].unique():
    tabletop_mask = (data['test_type'] == 'tabletop')
    data.loc[tabletop_mask, 'choices'] = data.loc[tabletop_mask, 'choices'].apply(
        lambda x: x if isinstance(x, str) and len(eval(x)) == 5 else np.nan
    )
if 'robot_nav' in data['test_type'].unique():
    robot_nav_mask = (data['test_type'] == 'robot_nav')
    data.loc[robot_nav_mask, 'choices'] = data.loc[robot_nav_mask, 'choices'].apply(
        lambda x: x if isinstance(x, str) and len(eval(x)) == 6 else np.nan
    )
data = data.dropna(subset=['choices'])

# Then filter for users who completed all conditions (0,1,2) for each test_type
# and have all survey metrics filled
valid_users = []
for user_id, user_data in data.groupby('user_id'):
    # Check if user has all conditions (0,1,2) for each test_type they attempted
    test_types = user_data['test_type'].unique()
    valid = True
    for test in test_types:
        if len(user_data[user_data['test_type'] == test]['condition_number'].unique()) < 3:
            valid = False
            break
    
    # Check all survey metrics are present and non-null
    if valid and not user_data[survey_metrics].isnull().any().any():
        valid_users.append(user_id)
        
print(f"Number of valid users: {len(valid_users)}")

data = data[data['user_id'].isin(valid_users)]

Number of valid users: 4


In [13]:
data[(data['test_type'] == 'tabletop') & (data['condition_number'] == 2)]['choices']

0     [1, 0, 1, 1, 0]
5     [1, 0, 0, 0, 1]
13    [1, 0, 1, 1, 0]
20    [1, 0, 1, 1, 1]
Name: choices, dtype: object

In [14]:
data[~data['decision_factors'].isnull()].sort_values(by='test_type').loc[:,['test_type', 'decision_factors']].to_csv('decision_factors.csv', index=False)

In [15]:
os.getcwd()

'/Users/weijiang/Documents/College/Lab/user-pref-survey/data analysis/tests'

In [16]:
class RemapUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module.startswith("numpy._core"):
            module = module.replace("numpy._core", "numpy.core", 1)
        return super().find_class(module, name)


def get_correlations(data: pd.DataFrame, condition_labels: List[str], outdir: str, tests: Optional[List[str]] = None):
    """
    Generate correlation box plots using PKL trees, driven by user choices from CSV.

    - Reads choices per (test_type, condition_number) from CSV
    - Loads the corresponding PKL tree for each condition
    - Traverses the first 5 choices to a final node
    - Computes estimated reward correlation for each user via get_estimated_rewards
    - Creates a seaborn box plot of correlations grouped by condition labels per test_type

    Args:
        data: pandas DataFrame (must include: test_type, condition_number, choices)
        condition_labels: labels in order of sorted unique condition_number values
        outdir: directory to write plots
        tests: optional subset of test_type values to include (e.g., ['robot_nav','tabletop'])
    """
    df = data

    tests_to_plot = tests if tests is not None and len(tests) > 0 else sorted(df['test_type'].unique())

    unique_conditions = sorted(df['condition_number'].unique())
    if len(condition_labels) != len(unique_conditions):
        raise ValueError(
            f"condition_labels length {len(condition_labels)} does not match number of unique conditions {len(unique_conditions)}"
        )
    condition_map = {cond: condition_labels[idx] for idx, cond in enumerate(unique_conditions)}

    # Resolve repo root to load PKL files reliably
    REPO_ROOT = "../../"

    os.makedirs(outdir, exist_ok=True)

    for test in tests_to_plot:
        sub = df[df['test_type'] == test]
        if sub.empty:
            raise ValueError(f"No rows for test: {test}")
        
        if test == "robot_nav":
            test_name = "GrassStreetNav"
            true_weights = GRASS_STREET_NAV_TRUE_WEIGHTS
            feature_bounds = GRASS_STREET_NAV_FEATURE_BOUNDS
        else:
            test_name = "TableTop"
            true_weights = TABLETOP_TRUE_WEIGHTS
            feature_bounds = TABLETOP_FEATURE_BOUNDS

        # Load trees per condition for this test_type
        trees_by_condition = {}
        for cond in unique_conditions:
            cond = int(cond)
            pkl_path = os.path.join(
                '..','..','server', 'assets', 'user_study', f'{test_name}-v{cond}', f'query_tree_{test_name}-v{cond}.pkl'
            )
            with open(pkl_path, 'rb') as f:
                trees_by_condition[cond] = RemapUnpickler(f).load()

        # Build per-user correlation rows for seaborn
        rows = []
        for _, row in sub.iterrows():
            cond = int(row['condition_number'])
            choices_str = row['choices']
            if isinstance(choices_str, str):
                try:
                    choices = ast.literal_eval(choices_str)
                except Exception:
                    raise ValueError(f"Invalid choices string: {choices_str}")
                if not isinstance(choices, list):
                    raise ValueError(f"Invalid choices type: {type(choices)}")
                if len(choices) < 5:
                    raise ValueError(f"Invalid choices length: {len(choices)}")

            final_node = trees_by_condition[cond]
            for idx, choice in enumerate(choices):
                if test == "robot_nav" and idx > 5:
                    break
                if test == "tabletop" and idx > 4:
                    break
                try:
                    final_node = final_node.children[choice]
                except Exception:
                    final_node = None
                    break
            if final_node is None:
                raise ValueError(f"Invalid final node: {final_node}")

            r, _ = get_estimated_rewards(final_node, true_weights, feature_bounds)
            rows.append({
                'test_type': test,
                'condition': condition_map[cond],
                'correlation': r,
                'choices': choices,
            })

        if not rows:
            raise ValueError(f"No rows for test: {test_name}")

        plot_df = pd.DataFrame(rows)
        plot_df.to_csv(os.path.join(outdir, f'correlation_{test_name}.csv'), index=False)

In [17]:
get_correlations(data, ["RL", "Envopt", "Fixed"], "out/")